In [60]:
import pandas as pd

In [61]:
df = pd.read_csv("../data/processed/model1_features.csv")

In [62]:
df.shape

(4958, 14)

In [63]:
y = df["days_to_payment"]

In [64]:
y.head()

0    66.0
1    71.0
2    67.0
3    62.0
4    73.0
Name: days_to_payment, dtype: float64

In [65]:
feature_columns = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_invoice_count",
    "customer_payment_std",
    "payment_behavior_trend",
    "previous_payment_days",
    "invoice_amount",
    "payment_term_days",
    "sector"
]

X = df[feature_columns]

In [66]:
X.head()

,customer_avg_payment_days,customer_recent_avg_payment_days,customer_invoice_count,customer_payment_std,payment_behavior_trend,previous_payment_days,invoice_amount,payment_term_days,sector
0,NaN,NaN,0,NaN,NaN,NaN,326004.92,60,Textiles & Apparel
1,66.0,66.000000,1,NaN,0.000000,66.0,194103.72,60,Textiles & Apparel
2,68.5,68.500000,2,3.535534,0.000000,71.0,331123.89,60,Textiles & Apparel
3,68.0,68.000000,3,2.645751,0.000000,67.0,191612.42,60,Textiles & Apparel
4,66.5,66.666667,4,3.696846,0.166667,62.0,305369.92,60,Textiles & Apparel


In [67]:
df["issue_date"].min(), df["issue_date"].max()

('2024-02-14', '2026-07-16')

In [68]:
df["issue_date"] = pd.to_datetime(df["issue_date"])

In [69]:
train_df = df[df["issue_date"] <= "2025-10-05"].copy()

validation_df = df[
    (df["issue_date"] >= "2025-10-06") &
    (df["issue_date"] <= "2026-02-14")
].copy()

test_df = df[df["issue_date"] >= "2026-02-15"].copy()

In [70]:
train_df.shape, validation_df.shape, test_df.shape

((3473, 14), (745, 14), (740, 14))

In [71]:
historical_features = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_payment_std",
    "payment_behavior_trend"
]

train_df[historical_features].median()

customer_avg_payment_days           47.500000
customer_recent_avg_payment_days    47.333333
customer_payment_std                 4.983903
payment_behavior_trend               0.000000
dtype: float64

In [72]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [73]:
numeric_features = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_invoice_count",
    "customer_payment_std",
    "payment_behavior_trend",
    "previous_payment_days",
    "invoice_amount",
    "payment_term_days"
]

categorical_features = [
    "sector"
]

In [74]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [75]:
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [76]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [77]:
target = "days_to_payment"

In [78]:
X_train = train_df[feature_columns]
y_train = train_df[target]

In [79]:
X_val = validation_df[feature_columns]
y_val = validation_df[target]

In [80]:
X_test = test_df[feature_columns]
y_test = test_df[target]

In [81]:
X_train.shape, X_val.shape, X_test.shape

((3473, 9), (745, 9), (740, 9))

In [82]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [83]:
X_train_processed = preprocessor.transform(X_train)

In [84]:
X_val_processed = preprocessor.transform(X_val)

In [85]:
X_test_processed = preprocessor.transform(X_test)

In [86]:
X_train_processed.shape, X_val_processed.shape, X_test_processed.shape

((3473, 18), (745, 18), (740, 18))

In [87]:
from sklearn.ensemble import RandomForestRegressor

In [88]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [89]:
rf_model.fit(X_train_processed, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [90]:
rf_val_predictions = rf_model.predict(X_val_processed)

In [91]:
rf_val_predictions[:10]

array([68.94 , 69.725, 68.155, 91.105, 92.635, 89.395, 88.21 , 91.825,
       91.71 , 89.87 ])

In [92]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [93]:
rf_val_mae = mean_absolute_error(y_val, rf_val_predictions)

In [94]:
rf_val_mae

7.2711409395973154

In [95]:
rf_val_rmse = mean_squared_error(y_val, rf_val_predictions) ** 0.5

In [96]:
rf_val_rmse

11.542508503368563

In [97]:
from xgboost import XGBRegressor

In [98]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [99]:
xgb_model.fit(
    X_train_processed,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [100]:
xgb_val_predictions = xgb_model.predict(X_val_processed)

In [101]:
xgb_val_predictions[:10]

array([69.7874  , 68.97469 , 65.91115 , 91.122826, 91.88333 , 90.555824,
       89.97554 , 90.180046, 92.47749 , 90.71085 ], dtype=float32)

In [102]:
xgb_val_mae = mean_absolute_error(y_val, xgb_val_predictions)

In [103]:
xgb_val_mae

7.888100199091355

In [104]:
xgb_val_rmse = mean_squared_error(y_val, xgb_val_predictions) ** 0.5

In [105]:
xgb_val_rmse

12.439481661653021

In [106]:
validation_results = validation_df[
    ["invoice_amount", "sector"]
].copy()

validation_results["actual_days"] = y_val.values
validation_results["predicted_days"] = rf_val_predictions

validation_results["absolute_error"] = (
    validation_results["actual_days"]
    - validation_results["predicted_days"]
).abs()

In [107]:
validation_results.head(10)

,invoice_amount,sector,actual_days,predicted_days,absolute_error
30,255398.76,Textiles & Apparel,66.0,68.940,2.940
31,213195.97,Textiles & Apparel,53.0,69.725,16.725
32,278884.01,Textiles & Apparel,66.0,68.155,2.155
63,25897.34,Textiles & Apparel,95.0,91.105,3.895
64,23027.47,Textiles & Apparel,91.0,92.635,1.635
65,28474.04,Textiles & Apparel,89.0,89.395,0.395
66,37567.64,Textiles & Apparel,95.0,88.210,6.790
67,17413.70,Textiles & Apparel,89.0,91.825,2.825
68,23514.95,Textiles & Apparel,90.0,91.710,1.710
69,28922.84,Textiles & Apparel,96.0,89.870,6.130


In [108]:
validation_results.sort_values(
    "absolute_error",
    ascending=False
).head(10)

,invoice_amount,sector,actual_days,predicted_days,absolute_error
1063,104233.19,Construction & Infra,66.0,140.020,74.020
1693,152441.23,Metal & Engineering,148.0,79.300,68.700
173,368430.19,Chemicals & Plastics,112.0,44.320,67.680
1062,73581.40,Construction & Infra,32.0,97.685,65.685
1687,150670.79,Metal & Engineering,40.0,87.785,47.785
1638,69128.87,Retail & Distribution,30.0,77.380,47.380
4362,361211.35,Construction & Infra,70.0,110.955,40.955
3029,59461.36,Construction & Infra,102.0,63.255,38.745
2845,431724.86,Pharma & Healthcare,85.0,47.280,37.720
1647,54993.00,Retail & Distribution,40.0,77.005,37.005


In [109]:
history_features = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_invoice_count",
    "customer_payment_std",
    "payment_behavior_trend",
    "payment_term_days"
]

error_analysis = validation_df[
    history_features
].copy()

error_analysis["actual_days"] = y_val.values
error_analysis["predicted_days"] = rf_val_predictions

error_analysis["absolute_error"] = (
    error_analysis["actual_days"]
    - error_analysis["predicted_days"]
).abs()

In [110]:
error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(10)

,customer_avg_payment_days,customer_recent_avg_payment_days,customer_invoice_count,customer_payment_std,payment_behavior_trend,payment_term_days,actual_days,predicted_days,absolute_error
1063,71.333333,39.666667,12,35.808413,-31.666667,45,66.0,140.020,74.020
1693,54.050000,52.333333,40,13.239316,-1.716667,45,148.0,79.300,68.700
173,40.103448,32.333333,29,20.091478,-7.770115,30,112.0,44.320,67.680
1062,74.909091,54.666667,11,35.237635,-20.242424,45,32.0,97.685,65.685
1687,54.911765,53.333333,34,14.074192,-1.578431,45,40.0,87.785,47.785
1638,56.763158,41.666667,38,27.654660,-15.096491,45,30.0,77.380,47.380
4362,98.677419,90.666667,31,13.644015,-8.010753,60,70.0,110.955,40.955
3029,57.478261,68.666667,23,14.951040,11.188406,30,102.0,63.255,38.745
2845,36.888889,38.000000,27,9.676829,1.111111,30,85.0,47.280,37.720
1647,56.276596,57.666667,47,25.505074,1.390071,45,40.0,77.005,37.005


In [111]:
error_analysis["archetype"] = validation_df["customer_archetype_TRUE_LABEL"].values

error_analysis.groupby("archetype")["absolute_error"].agg(["mean", "count"]).sort_values(
    "mean", ascending=False
)

,mean,count
archetype,,
erratic_payer,21.635000,72
deteriorating_payer,11.650078,64
cold_start,11.320833,12
improving_payer,8.014571,70
seasonal_payer,5.961307,88
average_payer,5.246845,206
chronic_late_payer,5.006579,57
prompt_payer,2.988551,176


In [112]:
validation_results[["actual_days", "predicted_days"]].describe()

,actual_days,predicted_days
count,745.000000,745.000000
mean,52.628188,54.322362
std,21.705074,20.883005
min,7.000000,12.965000
25%,37.000000,38.225000
50%,49.000000,49.575000
75%,64.000000,68.060000
max,148.000000,140.020000


In [113]:
validation_results["error"] = (
    validation_results["predicted_days"]
    - validation_results["actual_days"]
)

validation_results["error"].describe()

count    745.000000
mean       1.694174
std       11.425169
min      -68.700000
25%       -3.520000
50%        0.875000
75%        5.470000
max       74.020000
Name: error, dtype: float64